In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import get_isc_catalog, catalog_to_dataframe
from tqdm.auto import tqdm
from pathlib import Path
import time

In [3]:
EARTHQUAKE_REGIONS = {
    "california":       [[30, 43, -126, -113]],
    "cascadia":         [[40, 52, -132, -120]],
    "alaska":           [[50, 72, -170, -130]],

    "aleutians": [
        [48, 58, 160, 180],
        [48, 58, -180, -160],
    ],

    "mexico":           [[13, 33, -120, -85]],
    "central_america":  [[5, 20, -95, -75]],
    "caribbean":        [[8, 25, -90, -58]],

    "northern_andes":   [[-10, 15, -85, -65]],
    "central_andes":    [[-30, -8, -82, -62]],
    "southern_andes":   [[-58, -28, -80, -62]],

    "italy":            [[35, 48, 5, 20]],
    "aegean":           [[32, 43, 18, 32]],
    "turkiye":          [[35, 43, 25, 46]],
    "iran_zagros":      [[24, 40, 43, 63]],

    "hindu_kush_pamir": [[30, 40, 65, 78]],
    "himalaya":         [[25, 38, 72, 100]],

    "sumatra":          [[-8, 8, 92, 108]],
    "java":             [[-13, -3, 103, 116]],
    "philippines":      [[4, 22, 115, 130]],
    "taiwan":           [[20, 27, 118, 124]],
    "japan":            [[28, 46, 128, 148]],
    "kurils":           [[42, 52, 142, 160]],
    "kamchatka":        [[50, 62, 155, 170]],

    "tonga": [
        [-30, -12, 170, 180],
        [-30, -12, -180, -170],
    ],

    "new_zealand":      [[-48, -32, 165, 180]],
    "iceland":          [[62, 68, -26, -12]],
}

In [ ]:
start_year = 1995
end_year = 2026
min_magnitude = 3.0
overwrite_existing = False
pbar = tqdm(total=(end_year - start_year + 1) * len(EARTHQUAKE_REGIONS), desc="Downloading ISC catalogs")
for year in range(start_year, end_year + 1):
    for region, bounding_boxes in EARTHQUAKE_REGIONS.items():
        pbar.update(1)
        target_catalog_path = Path(f"catalogs/isc/{region}_{year}.csv")
        target_mag_catalog_path = Path(f"catalogs/isc_mag/{region}_{year}_magnitudes.csv")
        if not overwrite_existing and target_catalog_path.exists() and target_mag_catalog_path.exists():
            continue
        region_year_catalog = []
        region_mag_year_catalog = []
        error_occurred = False
        for bounding_box in bounding_boxes:
            
                for month in range(1, 13):
                    start_time = f"{year}-{month:02d}-01"
                    if month < 12:
                        end_time = f"{year}-{month+1:02d}-01"
                    else:
                        end_time = f"{year+1}-01-01"
                    pbar.set_postfix({"Year": year, "Month" : month, "Region": region})
                    
                    try:
                        catalog = get_isc_catalog(
                            start_time=start_time,
                            end_time=end_time,
                            bounding_box=bounding_box,
                            min_magnitude=min_magnitude,
                            max_magnitude=None,
                            include_all_magnitudes=True
                        )
                    except Exception as e:
                        print(f"Error occurred while fetching catalog for {region} in {year}: {e}")
                        error_occurred = True
                        continue
                    time.sleep(0.5)
                    catalog_df, mag_df = catalog_to_dataframe(catalog, include_all_magnitudes=True)
                    region_year_catalog.append(catalog_df)
                    region_mag_year_catalog.append(mag_df)
        if len(region_year_catalog) == 0 or len(region_mag_year_catalog) == 0 or error_occurred:
            print(f"No data fetched for {region} in {year}. Skipping saving.")
            continue
        region_year_catalog_df = pd.concat(region_year_catalog, ignore_index=True)
        region_mag_year_catalog_df = pd.concat(region_mag_year_catalog, ignore_index=True)
        region_year_catalog_df.sort_values(by='time', inplace=True)
        region_year_catalog_df.to_csv(target_catalog_path, index=False)
        region_mag_year_catalog_df.to_csv(target_mag_catalog_path, index=False)

In [44]:
from torch.utils.data import Dataset
class CatalogDataset(Dataset):
    def __init__(self, duration_min_mean : float, duration_min_std : float,
                catalog_df : pd.DataFrame, magnitude_df : Optional[pd.DataFrame] = None, 
                event_id_name : str = 'event_id', time_name : str = 'time', 
                latitude_name : str = 'latitude', longitude_name : str = 'longitude', depth_name : str = 'depth', 
                magnitude_name : str = 'magnitude', magnitude_type : str = 'magnitude_type', magnitude_family : str = "magnitude_family"):
        self.event_id_name = event_id_name
        self.time_name = time_name
        self.latitude_name = latitude_name
        self.longitude_name = longitude_name
        self.depth_name = depth_name
        self.magnitude_name = magnitude_name
        self.magnitude_type = magnitude_type
        self.magnitude_family = magnitude_family
        self.catalog_df = catalog_df
        self.magnitude_df = magnitude_df
        self.return_all_magnitudes = magnitude_df is not None
        self.duration_mean = duration_min_mean
        self.duration_std = duration_min_std
        self.duration_scale = (self.duration_std**2)/self.duration_mean
        self.duration_shape = self.duration_mean/self.duration_scale
        
    def __len__(self):
        return len(self.catalog_df)


    
    def __getitem__(self, idx):
        duration = np.random.gamma(self.duration_shape, self.duration_scale)
        start_time = self.catalog_df.iloc[idx][self.time_name].to_numpy()
        end_time = start_time + np.timedelta64(int(duration*60), 's')
        mask = (self.catalog_df[self.time_name] <= end_time) & (self.catalog_df[self.time_name] >= start_time)
        num_events = mask.sum()
        catalog_subset = self.catalog_df[mask]
        times_from_start = (catalog_subset[self.time_name].to_numpy() - start_time)
        locations = np.zeros((num_events, 3), dtype=np.float32)
        locations[:, 0] = catalog_subset[self.latitude_name].to_numpy()
        locations[:, 1] = catalog_subset[self.longitude_name].to_numpy()
        locations[:, 2] = catalog_subset[self.depth_name].to_numpy()
        magnitudes = catalog_subset[self.magnitude_name].to_numpy()
        return start_time, times_from_start, magnitudes, locations


        
    